# Ejercicio: Asistente de Documentos con Gemini y Gradio
 
**Tiempo:** 40-45 minutos  
**Grupos:** 3-4 personas  
 
## Contexto
 
En los notebooks anteriores construimos chatbots que responden preguntas generales.
En este ejercicio daremos un paso más: construir un asistente que responda preguntas
sobre un documento específico — en este caso, el paper seminal de los Transformers:
**"Attention is All You Need"** (Vaswani et al., 2017).
 
Esta técnica — inyectar el contenido de un documento en el contexto del modelo —
es la base conceptual de **RAG (Retrieval-Augmented Generation)**, uno de los
patrones más usados en aplicaciones de IA en producción.
 
## Objetivo
 
Construir una aplicación Gradio donde el usuario pueda:
1. Cargar el PDF del paper
2. Hacer preguntas sobre su contenido
3. Obtener respuestas basadas **exclusivamente** en el documento
 
## Lo que van a aprender
 
- Extracción de texto desde PDFs con `pypdf`
- Inyección de contexto externo en el system prompt
- Limitaciones de este enfoque y por qué existe RAG
- Integración de `gr.File` en una interfaz Gradio
 
---

## Paso 0: Instalación y configuración
 
### 0.1 Instala las dependencias necesarias
 
Necesitarás tres bibliotecas nuevas además de las que ya conoces:
- `pypdf`: para extraer texto de archivos PDF
- `gradio`: para la interfaz web
- `google-genai`: para el modelo
 
```
pip install pypdf gradio google-genai python-dotenv
```

In [1]:
%pip install pypdf gradio google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### 0.2 Descarga el paper
 
Descarga el PDF de ArXiv (acceso abierto):
```
https://arxiv.org/pdf/1706.03762
```
 
Guárdalo en la misma carpeta que este notebook con el nombre `attention_is_all_you_need.pdf`.
 
### 0.3 Configura tus credenciales
 
Crea un archivo `.env` con tu API key de Gemini:
```
GEMINI_API_KEY="tu_key_aqui"
```

## Paso 1: Extracción de texto del PDF
 
Lo primero es leer el PDF y extraer su contenido como texto plano.
`pypdf` hace esto en pocas líneas.
 
**Instrucciones:**
1. Importa `PdfReader` desde `pypdf`
2. Crea una función `extract_text_from_pdf(pdf_path)` que:
   - Abra el PDF desde la ruta `pdf_path`
   - Itere sobre todas las páginas
   - Concatene el texto de cada página
   - Retorne el texto completo como string
3. Prueba la función con `attention_is_all_you_need.pdf`
4. Imprime los primeros 500 caracteres para verificar que funcionó
 
**Pista:** `PdfReader` tiene un atributo `pages` que es una lista.
Cada página tiene un método `extract_text()`.
"""

In [2]:
from pypdf import PdfReader

def extract_text_from_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = ""
    for page in reader.pages:
        text += page.extract_text()
    return text

#proabmos la función
text = extract_text_from_pdf("attention_is_all_you_need.pdf")
print(text[:500])  # Imprime los primeros 500 caracteres del libro

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz 


## Paso 2: Inicialización del cliente de Gemini
 
Igual que en los notebooks anteriores.
 
**Instrucciones:**
1. Importa las bibliotecas necesarias (`os`, `dotenv`, `google.genai`, `google.genai.types`)
2. Carga las variables de entorno
3. Inicializa el cliente de Gemini
4. Define la constante `MODELO = "gemini-2.5-flash-lite"`
5. Verifica que la API key esté disponible
"""

In [3]:
import os
import gradio as gr
from google import genai
from google.genai import types
from dotenv import load_dotenv

load_dotenv()

if os.getenv("GEMINI_API_KEY"):
    print("Gemini API Key cargada correctamente")
else:
    print("ERROR: GEMINI_API_KEY no encontrada. Verifica tu archivo .env")

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

MODELO = "gemini-2.5-flash-lite"

c:\Users\USUARIO\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gemini API Key cargada correctamente


## Paso 3: Función de chat con contexto del documento
 
Esta es la parte central del ejercicio. La idea es construir un system prompt
que incluya el texto completo del paper, instruyendo al modelo a responder
**solo** basándose en ese contenido.
 
**Instrucciones:**
1. Crea una función `build_system_prompt(document_text)` que reciba el texto
   del documento y retorne un system prompt que:
   - Defina el rol del asistente (experto en el paper)
   - Incluya el texto completo del documento
   - Instruya al modelo a responder **solo** con información del documento
   - Indique qué hacer si la respuesta no está en el documento
 
2. Crea una función `chat_con_documento(message, history, document_text)` que:
   - Construya el historial en formato Gemini (objetos `types.Content`)
   - Use `generate_content_stream` con el system prompt del documento
   - Retorne la respuesta con `yield` para streaming
 
**Pista:** Recuerda que en Gradio el historial llega como lista de dicts
con claves `role` y `content`. El rol del asistente en Gemini es `"model"`.
"""

In [ ]:
def build_system_prompt(document_text):
    return f"""Eres un asistente experto en el paper académico que se te proporciona a continuación.
Tu única fuente de información es el contenido del documento.

REGLAS:
- Responde ÚNICAMENTE con información que esté explícitamente en el documento.
- Si la pregunta no puede responderse con el documento, di exactamente: 
  "Esa información no se encuentra en el documento."
- No uses tu conocimiento general ni información externa.

DOCUMENTO:
{document_text}
"""

def chat_con_documento(message, history, document_text):
    system_prompt = build_system_prompt(document_text)

    # Construir historial en formato Gemini
    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else "user"
        gemini_history.append(
            types.Content(role=role, parts=[types.Part(text=msg["content"])])
        )

    # Agregar mensaje actual del usuario
    gemini_history.append(
        types.Content(role="user", parts=[types.Part(text=message)])
    )

    # Llamada con streaming
    response = client.models.generate_content_stream(
        model=MODELO,
        contents=gemini_history,
        config=types.GenerateContentConfig(system_instruction=system_prompt)
    )

    partial = ""
    for chunk in response:
        if chunk.text:
            partial += chunk.text
            yield partial

## Paso 4: Interfaz Gradio
 
Ahora construimos la interfaz. Usaremos `gr.ChatInterface` con `additional_inputs`
para pasar el texto del documento extraído.
 
**Instrucciones:**
1. Extrae el texto del PDF usando la función del Paso 1
2. Imprime cuántas páginas tiene y cuántos caracteres extraíste
3. Construye la interfaz con `gr.ChatInterface`:
   - `fn`: tu función `chat_con_documento`
   - `title`: un título descriptivo
   - `description`: explica qué puede hacer el asistente
   - `additional_inputs`: un `gr.Textbox` visible=False que contenga el texto
     del documento (así Gradio lo pasa automáticamente a la función)
   - `examples`: al menos 3 preguntas relevantes sobre el paper
 
**Pista:** Para pasar el texto del documento sin mostrarlo en la interfaz:
```python
gr.Textbox(value=document_text, visible=False)
```
 
4. Lanza la interfaz con `demo.launch(server_name="0.0.0.0", server_port=8080)`
"""

In [ ]:
# Extraer texto del PDF
document_text = extract_text_from_pdf("attention_is_all_you_need.pdf")
print(f"Páginas: {len(PdfReader('attention_is_all_you_need.pdf').pages)}")
print(f"Caracteres extraídos: {len(document_text)}")

demo = gr.ChatInterface(
    fn=chat_con_documento,
    title="📄 Asistente: Attention is All You Need",
    description="Haz preguntas sobre el paper seminal de los Transformers (Vaswani et al., 2017). Las respuestas se basan exclusivamente en el contenido del documento.",
    additional_inputs=[
        gr.Textbox(value=document_text, visible=False)
    ],
    examples=[
        ["¿Cuál es la arquitectura principal propuesta en el paper?"],
        ["¿Qué es el mecanismo de atención y cómo funciona?"],
        ["¿Cuántas capas tiene el encoder del modelo base?"],
        ["¿Cuál es el resultado en la tarea WMT 2014 English-to-German?"],
        ["¿Quiénes son los autores del paper?"],
    ]
)

demo.launch(server_name="0.0.0.0", server_port=8080)

In [ ]:
#demo.close()

## Paso 5: Prueba y reflexión
 
Una vez que la interfaz esté funcionando, prueba estas preguntas:
 
1. *"¿Cuál es la arquitectura principal propuesta en el paper?"*
2. *"¿Qué es el mecanismo de atención?"*
3. *"¿Cuántas capas tiene el encoder del modelo base?"*
4. *"¿Quiénes son los autores del paper?"*
5. *"¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?"*
 
Y esta pregunta trampa:
6. *"¿Qué es GPT-4?"*
 
Esta última pregunta **no está en el paper**. Observa cómo responde el modelo.
¿Usa su conocimiento general o respeta la instrucción de ceñirse al documento?
 

In [ ]:
preguntas_prueba = [
    "¿Cuál es la arquitectura principal propuesta en el paper?",
    "¿Qué es el mecanismo de atención?",
    "¿Cuántas capas tiene el encoder del modelo base?",
    "¿Quiénes son los autores del paper?",
    "¿Cuál es el resultado del modelo en la tarea WMT 2014 English-to-German?",
    "¿Qué es GPT-4?",
]

for pregunta in preguntas_prueba:
    print(f"\nPregunta: {pregunta}")
    respuesta = ""
    for parcial in chat_con_documento(pregunta, [], document_text):
        respuesta = parcial
    print(f"Respuesta: {respuesta}\n")

## Paso 6 (Adicional): Mejora el sistema
 
Si terminaron antes de tiempo, implementen **una** de estas mejoras:
 
**A) Indicador de tokens**
Muestra cuántos tokens está usando el system prompt. Recuerda que Gemini 2.5 Flash
tiene un límite de 1,000,000 tokens — ¿qué porcentaje del límite usa este paper?
 
**B) Subida dinámica de PDF**
En lugar de cargar el PDF con código, agrega un `gr.File` en `additional_inputs`
para que el usuario pueda subir cualquier PDF desde la interfaz.
Tendrás que extraer el texto dentro de la función de chat cuando llegue el archivo.
 
**C) Citas del documento**
Modifica el system prompt para que el modelo siempre incluya una cita textual
del paper que respalde su respuesta.

In [ ]:
# Paso 6 completo
# A) Indicador de tokens
# B) Subida dinámica de PDF
# C) Citas del documento en cada respuesta

LIMITE_TOKENS_MODELO = 1_000_000
_pdf_cache = {}


def extract_text_safe(pdf_path):
    """Extrae texto de un PDF y evita fallos si alguna página viene vacía."""
    reader = PdfReader(pdf_path)
    pages_text = []
    for page in reader.pages:
        pages_text.append(page.extract_text() or "")
    return "\n".join(pages_text), len(reader.pages)


def build_system_prompt_con_citas(document_text):
    return f"""Eres un asistente experto en el documento proporcionado.
Tu única fuente de información es el texto del documento incluido abajo.

REGLAS OBLIGATORIAS:
1) Responde ÚNICAMENTE con información explícita del documento.
2) Si no hay evidencia suficiente, responde exactamente:
   "Esa información no se encuentra en el documento."
3) No uses conocimiento externo.
4) Siempre incluye al final una cita textual breve del documento entre comillas,
   precedida por: Cita del documento:

DOCUMENTO:
{document_text}
"""


def contar_tokens_prompt(system_prompt):
    """Cuenta tokens del system prompt y calcula porcentaje del límite."""
    conteo = client.models.count_tokens(
        model=MODELO,
        contents=[types.Content(role="user", parts=[types.Part(text=system_prompt)])],
    )
    tokens_usados = conteo.total_tokens
    porcentaje = (tokens_usados / LIMITE_TOKENS_MODELO) * 100
    return tokens_usados, porcentaje


def preparar_documento(pdf_file):
    """Carga y cachea texto del PDF subido en Gradio (type='filepath')."""
    if not pdf_file:
        return None, "Primero sube un archivo PDF en el campo 'PDF del documento'."

    pdf_path = pdf_file  # gr.File(type='filepath') entrega una ruta str

    if pdf_path in _pdf_cache:
        return _pdf_cache[pdf_path], None

    try:
        document_text_local, total_paginas = extract_text_safe(pdf_path)
        if not document_text_local.strip():
            return None, "No se pudo extraer texto legible del PDF."

        system_prompt_local = build_system_prompt_con_citas(document_text_local)
        tokens_usados, porcentaje = contar_tokens_prompt(system_prompt_local)

        info = {
            "pdf_path": pdf_path,
            "document_text": document_text_local,
            "total_paginas": total_paginas,
            "tokens_usados": tokens_usados,
            "porcentaje_limite": porcentaje,
            "system_prompt": system_prompt_local,
        }
        _pdf_cache[pdf_path] = info
        return info, None
    except Exception as error:
        return None, f"Error procesando el PDF: {error}"


def chat_con_documento_dinamico(message, history, pdf_file):
    info, error = preparar_documento(pdf_file)
    if error:
        yield error
        return

    gemini_history = []
    for msg in history:
        role = "model" if msg["role"] == "assistant" else "user"
        gemini_history.append(
            types.Content(role=role, parts=[types.Part(text=msg["content"])])
        )

    gemini_history.append(
        types.Content(role="user", parts=[types.Part(text=message)])
    )

    encabezado = (
        f"[Indicador de tokens] Prompt: {info['tokens_usados']} tokens "
        f"({info['porcentaje_limite']:.4f}% del límite de {LIMITE_TOKENS_MODELO}).\n\n"
    )

    try:
        response = client.models.generate_content_stream(
            model=MODELO,
            contents=gemini_history,
            config=types.GenerateContentConfig(system_instruction=info["system_prompt"])
        )

        partial = encabezado
        for chunk in response:
            if chunk.text:
                partial += chunk.text
                yield partial
    except Exception as error:
        yield f"Error al consultar Gemini: {error}"


# Cierra la demo anterior si estaba abierta
try:
    demo.close()
except Exception:
    pass


# Interfaz mejorada con subida dinámica de PDF
# Opción B implementada con gr.File en additional_inputs
# Opción A visible en cada respuesta (encabezado de tokens)
# Opción C aplicada vía system prompt con cita obligatoria
demo = gr.ChatInterface(
    fn=chat_con_documento_dinamico,
    title="Asistente de documentos",
    description=(
        "Sube un PDF y haz preguntas sobre su contenido. "
        "La respuesta usa solo el documento, muestra indicador de tokens "
        "y exige una cita textual del PDF."
    ),
    additional_inputs=[
        gr.File(label="PDF del documento", file_types=[".pdf"], type="filepath")
    ],
    examples=[
        ["¿Cuál es la contribución principal del documento?", None],
        ["Resume la metodología en 4 puntos.", None],
        ["¿Qué limitaciones menciona el documento?", None],
    ],
)

# Usa 8081 para evitar conflicto si 8080 ya está ocupado
demo.launch(server_name="0.0.0.0", server_port=8081)

* Running on local URL:  http://0.0.0.0:8081
* To create a public link, set `share=True` in `launch()`.


## Reflexión final
 
Discutan en grupo:
 
1. **¿Cuál es la limitación principal de este enfoque?**  
   Piensen en qué pasaría con un documento de 1000 páginas.

   **Respuesta:**  
   La limitación principal es la **escalabilidad del contexto**. En este ejercicio estamos insertando prácticamente todo el documento dentro del system prompt, lo cual funciona para un paper relativamente corto, pero se vuelve frágil cuando el volumen crece.

   Cuando el documento es muy grande, aparecen varios problemas al mismo tiempo:
   - **Costo:** más texto de entrada implica más tokens procesados y, por tanto, mayor costo por consulta.
   - **Latencia:** el modelo tarda más en leer y razonar sobre un contexto enorme antes de responder.
   - **Límite de contexto:** aunque algunos modelos aceptan muchas entradas, existe un tope; si se supera, parte del contenido no entra.
   - **Ruido informativo:** incluir "todo" hace más difícil enfocarse en lo relevante y puede degradar la precisión.

   En resumen, este enfoque es útil como prototipo didáctico, pero no es eficiente para colecciones grandes o uso en producción.
 
2. **¿Por qué existe RAG?**  
   ¿Cómo resolvería RAG el problema que identificaron?

   **Respuesta:**  
   RAG (Retrieval-Augmented Generation) existe para separar dos tareas que aquí están mezcladas: **buscar información** y **generar respuesta**.

   En lugar de pasar el documento completo cada vez, RAG normalmente hace este flujo:
   1. Divide los documentos en fragmentos (chunks).
   2. Convierte esos fragmentos en embeddings y los indexa en una base vectorial.
   3. Ante una pregunta, recupera solo los fragmentos más relevantes.
   4. Envía al modelo únicamente esos fragmentos junto con la pregunta.

   Esto resuelve los problemas anteriores porque:
   - Reduce tokens por consulta y por tanto costo.
   - Mejora tiempos de respuesta.
   - Permite trabajar con miles de documentos sin meter todo al prompt.
   - Aumenta trazabilidad, ya que se puede mostrar de qué fragmentos salió la respuesta.

   Por eso RAG no es solo una optimización técnica; también es una estrategia de calidad y gobernanza de respuestas.
 
3. **¿Qué información podría "filtrarse" aunque el system prompt diga que no?**  
   El modelo tiene conocimiento previo sobre el paper — ¿cómo verificarían
   que está respondiendo desde el documento y no desde su entrenamiento?
   
   **Respuesta:**  
   Lo que puede filtrarse es el **conocimiento paramétrico previo** del modelo: datos aprendidos durante el entrenamiento (por ejemplo, detalles del paper, contexto histórico o interpretaciones comunes) que no necesariamente están en el archivo cargado.

   Para verificar que la respuesta proviene del documento y no de memoria del modelo, se puede aplicar un protocolo simple:
   1. **Exigir evidencia textual:** pedir una cita literal o parafraseo con referencia a sección/página.
   2. **Verificación cruzada:** comparar la respuesta con el fragmento exacto del PDF.
   3. **Pruebas de control:** hacer preguntas fuera de alcance (como "¿Qué es GPT-4?") y confirmar rechazo.
   4. **Pruebas de contradicción:** introducir preguntas con premisas falsas para ver si corrige usando el texto.
   5. **Auditoría de trazabilidad:** guardar pregunta, contexto enviado y respuesta para revisión posterior.

   Una buena práctica final es definir una política de respuesta: si no hay evidencia en el documento, el asistente debe declarar explícitamente que no puede responder con certeza.
---


 
## Recursos
 
- [Documentación de pypdf](https://pypdf.readthedocs.io)
- [Documentación de Gradio](https://www.gradio.app/docs)
- [Documentación de Gemini API](https://ai.google.dev/gemini-api/docs)
- [Paper original en ArXiv](https://arxiv.org/abs/1706.03762)
"""